In [4]:
from pathlib import Path
import numpy as np
import cv2
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

2026-03-04 15:36:58.364773: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-04 15:36:59.151490: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-04 15:37:06.076229: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [ ]:
DATA_DIR = "/content/"
IMG_SIZE = (224, 224)

In [ ]:
def load_images_and_labels(folder):
    paths = Path(folder).glob("*.jpg")
    imgs = []
    labels = []

    for p in paths:
        img = cv2.imread(str(p))
        imgs.append(img)
        label = str(p).split("_")[-1].split(".")[0]
        labels.append(int(label))

    return imgs, labels

In [ ]:
imgs, labels = load_images_and_labels(DATA_DIR)

In [3]:
len(imgs)

NameError: name 'imgs' is not defined

In [ ]:
len(labels)

In [ ]:
def preprocess_images(imgs, target_size=IMG_SIZE):
  out = [cv2.resize(img, target_size) for img in imgs]
  out = np.asarray(out) / 255
  return out

In [ ]:
X = preprocess_images(imgs)

In [ ]:
y = np.asarray(labels)

In [ ]:
plt.imshow(X[0, :, :, ::-1])

In [ ]:
def show_examples(images, labels, n=8, title="Beispiele"):
    n = min(n, len(images))
    idx = np.random.choice(len(images), n, replace=False)
    cols = 4
    rows = int(np.ceil(n / cols))
    plt.figure(figsize=(12, 3*rows))
    for i, j in enumerate(idx, 1):
        plt.subplot(rows, cols, i)
        img = images[j]
        if img.dtype != np.uint8:
            img = (np.clip(img, 0, 1) * 255).astype(np.uint8)
        plt.imshow(img[:,:,::-1]); plt.title(f"Label: {labels[j]}"); plt.axis("off")
    plt.suptitle(title); plt.tight_layout(); plt.show()

In [ ]:
show_examples(X, y)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
def mirror_images(images, labels):
    X_m = [img[:,::-1,:] for img in images]
    y_m = [180 - label for label in labels]

    return X_m, y_m

In [ ]:
X_train_m, y_train_m = mirror_images(X_train, y_train)
X_train_pool = np.concatenate([X_train, X_train_m])
y_train_pool = np.concatenate([y_train, y_train_m])

In [ ]:
X_train_pool.shape

In [ ]:
plt.hist(y_test)

In [ ]:
size = X_train_pool.shape[1:]

In [ ]:
model = tf.keras.Sequential(name='Fahrspurerkennung')
model.add(layers.Resizing(480,640))
# Convolution Layers 
# elu: Expenential Linear Unit, similar to leaky Relu 
model.add(layers.Conv2D(24, (5, 5), strides=(2, 2), input_shape=(100, 200, 3), activation='elu')) 
model.add(layers.Conv2D(36, (5, 5), strides=(2, 2), activation='elu'))
model.add(layers.Conv2D(48, (5, 5), strides=(2, 2), activation='elu'))
model.add(layers.Conv2D(64, (3, 3), activation='elu')) 
model.add(layers.Dropout(0.2)) # more robustness 
model.add(layers.Conv2D(64, (3, 3), activation='elu')) 
# Fully Connected Layers 
model.add(layers.Flatten()) 
model.add(layers.Dropout(0.2)) # more robustness 
model.add(layers.Dense(100, activation='elu')) 
model.add(layers.Dense(50, activation='elu')) 
model.add(layers.Dense(10, activation='elu'))
# Output Layer: turning angle
model.add(layers.Dense(1)) 

In [ ]:
model.compile(optimizer="adam", loss="mse", metrics=["mae"])
history = model.fit(X_train_pool, y_train_pool, epochs=10, batch_size=32, validation_data=(X_test, y_test), verbose=1)

In [ ]:
# History-Plots
plt.figure(figsize=(6,4))
plt.plot(history.history["loss"], label="train_loss")
plt.plot(history.history["val_loss"], label="val_loss")
plt.xlabel("Epoch"); plt.ylabel("MSE"); plt.title("Training History (Loss)")
plt.legend(); plt.grid(True, linestyle="--", alpha=0.4); plt.tight_layout(); plt.show()

plt.figure(figsize=(6,4))
plt.plot(history.history["mae"], label="train_mae")
plt.plot(history.history["val_mae"], label="val_mae")
plt.xlabel("Epoch"); plt.ylabel("MAE"); plt.title("Training History (MAE)")
plt.legend(); plt.grid(True, linestyle="--", alpha=0.4); plt.tight_layout(); plt.show()

In [6]:
error = abs(y_test - y_pred)

NameError: name 'y_test' is not defined

In [ ]:
error

''

In [ ]:
np.argmax(error)

In [ ]:
plt.imshow(X_test[4, :, :, ::-1])

In [ ]:
model.save("model_peter.keras")

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()
with open("peter_cnn.tflite", "wb") as f:
  f.write(tflite_model)

In [ ]:
#Für Pi:
interpreter = tflite.Interpreter(model_path="/home/pi/Desktop/git/c2c_Phase2/Software/live_model_fp32.tflite")
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()
interpreter.allocate_tensors()
img = np.expand_dims(img, axis=0)
interpreter.set_tensor(input_details[0]['index'], img)
interpreter.invoke()
output_data = interpreter.get_tensor(output_details[0]['index'])
int(output_data)